In [2]:
import gmsh

gmsh.initialize()
gmsh.model.add("five_layers_two_wells")

# --------------------------------------------------
# 1. GEOMETRY PARAMETERS
# --------------------------------------------------
Lx = 10.0
Ly = 10.0

# 5 layers in z
z0 = 0.0
z1 = 1.5
z2 = 2.0
z3 = 3.0
z4 = 3.5
z5 = 5.0

# Central layer = layer 3 = between z2 and z3
z_middle_central = (z2 + z3) / 2.0   # 5.0

# Well radius
rw = 0.2

# Opposite-corner well positions
# Slightly moved inward so they are not exactly on the external boundary
xw1, yw1 = 1.0, 1.0
xw2, yw2 = 9.0, 9.0

# --------------------------------------------------
# 2. CREATE 5 LAYERED BOXES
# --------------------------------------------------
layer1 = gmsh.model.occ.addBox(0, 0, z0, Lx, Ly, z1 - z0)
layer2 = gmsh.model.occ.addBox(0, 0, z1, Lx, Ly, z2 - z1)
layer3 = gmsh.model.occ.addBox(0, 0, z2, Lx, Ly, z3 - z2)
layer4 = gmsh.model.occ.addBox(0, 0, z3, Lx, Ly, z4 - z3)
layer5 = gmsh.model.occ.addBox(0, 0, z4, Lx, Ly, z5 - z4)

# --------------------------------------------------
# 3. CREATE 2 WELLS
#    From top surface down to middle of central layer
# --------------------------------------------------
well_length = z5 - z_middle_central   # from z=10 down to z=5

well1 = gmsh.model.occ.addCylinder(xw1, yw1, z5, 0, 0, -well_length, rw)
well2 = gmsh.model.occ.addCylinder(xw2, yw2, z5, 0, 0, -well_length, rw)

# --------------------------------------------------
# 4. FRAGMENT EVERYTHING
#    This makes wells conform with the mesh
# --------------------------------------------------
domain_volumes = [(3, layer1), (3, layer2), (3, layer3), (3, layer4), (3, layer5)]
well_volumes = [(3, well1), (3, well2)]

gmsh.model.occ.fragment(domain_volumes, well_volumes)
gmsh.model.occ.synchronize()

# --------------------------------------------------
# 5. GLOBAL MESH SIZES
#    We first assign a general size to all points
# --------------------------------------------------
all_points = gmsh.model.getEntities(0)
gmsh.model.mesh.setSize(all_points, 10)

# --------------------------------------------------
# 6. REFINEMENT BY LAYER USING BOX FIELDS
#    Small size = more refined
# --------------------------------------------------
# Layer 1 and 5: coarsest
field1 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field1, "VIn", 10)
gmsh.model.mesh.field.setNumber(field1, "VOut", 10)
gmsh.model.mesh.field.setNumber(field1, "XMin", 0)
gmsh.model.mesh.field.setNumber(field1, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field1, "YMin", 0)
gmsh.model.mesh.field.setNumber(field1, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field1, "ZMin", z0)
gmsh.model.mesh.field.setNumber(field1, "ZMax", z1)

field5 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field5, "VIn", 10)
gmsh.model.mesh.field.setNumber(field5, "VOut", 10)
gmsh.model.mesh.field.setNumber(field5, "XMin", 0)
gmsh.model.mesh.field.setNumber(field5, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field5, "YMin", 0)
gmsh.model.mesh.field.setNumber(field5, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field5, "ZMin", z4)
gmsh.model.mesh.field.setNumber(field5, "ZMax", z5)

# Layer 2 and 4: intermediate
field2 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field2, "VIn", 2)
gmsh.model.mesh.field.setNumber(field2, "VOut", 10)
gmsh.model.mesh.field.setNumber(field2, "XMin", 0)
gmsh.model.mesh.field.setNumber(field2, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field2, "YMin", 0)
gmsh.model.mesh.field.setNumber(field2, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field2, "ZMin", z1)
gmsh.model.mesh.field.setNumber(field2, "ZMax", z2)

field4 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field4, "VIn", 2)
gmsh.model.mesh.field.setNumber(field4, "VOut", 10)
gmsh.model.mesh.field.setNumber(field4, "XMin", 0)
gmsh.model.mesh.field.setNumber(field4, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field4, "YMin", 0)
gmsh.model.mesh.field.setNumber(field4, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field4, "ZMin", z3)
gmsh.model.mesh.field.setNumber(field4, "ZMax", z4)

# Layer 3: most refined
field3 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field3, "VIn", 1)
gmsh.model.mesh.field.setNumber(field3, "VOut", 2)
gmsh.model.mesh.field.setNumber(field3, "XMin", 0)
gmsh.model.mesh.field.setNumber(field3, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field3, "YMin", 0)
gmsh.model.mesh.field.setNumber(field3, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field3, "ZMin", z2)
gmsh.model.mesh.field.setNumber(field3, "ZMax", z3)

# --------------------------------------------------
# 7. EXTRA REFINEMENT AROUND WELLS
# --------------------------------------------------
well_field1 = gmsh.model.mesh.field.add("Cylinder")
gmsh.model.mesh.field.setNumber(well_field1, "VIn", 1)
gmsh.model.mesh.field.setNumber(well_field1, "VOut", 10)
gmsh.model.mesh.field.setNumber(well_field1, "XCenter", xw1)
gmsh.model.mesh.field.setNumber(well_field1, "YCenter", yw1)
gmsh.model.mesh.field.setNumber(well_field1, "ZCenter", z_middle_central)
gmsh.model.mesh.field.setNumber(well_field1, "XAxis", 0)
gmsh.model.mesh.field.setNumber(well_field1, "YAxis", 0)
gmsh.model.mesh.field.setNumber(well_field1, "ZAxis", well_length)
gmsh.model.mesh.field.setNumber(well_field1, "Radius", 0.8 * 1.5)

well_field2 = gmsh.model.mesh.field.add("Cylinder")
gmsh.model.mesh.field.setNumber(well_field2, "VIn", 1)
gmsh.model.mesh.field.setNumber(well_field2, "VOut", 10)
gmsh.model.mesh.field.setNumber(well_field2, "XCenter", xw2)
gmsh.model.mesh.field.setNumber(well_field2, "YCenter", yw2)
gmsh.model.mesh.field.setNumber(well_field2, "ZCenter", z_middle_central)
gmsh.model.mesh.field.setNumber(well_field2, "XAxis", 0)
gmsh.model.mesh.field.setNumber(well_field2, "YAxis", 0)
gmsh.model.mesh.field.setNumber(well_field2, "ZAxis", well_length)
gmsh.model.mesh.field.setNumber(well_field2, "Radius", 0.8 * 1.5)

# --------------------------------------------------
# 8. COMBINE FIELDS
#    Use the minimum size from all refinement fields
# --------------------------------------------------
min_field = gmsh.model.mesh.field.add("Min")
gmsh.model.mesh.field.setNumbers(
    min_field,
    "FieldsList",
    [field1, field2, field3, field4, field5, well_field1, well_field2]
)
gmsh.model.mesh.field.setAsBackgroundMesh(min_field)

# Optional: make Gmsh respect the background field better
gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)

# --------------------------------------------------
# 9. GENERATE 3D MESH
# --------------------------------------------------
gmsh.model.mesh.generate(3)

# --------------------------------------------------
# 10. SAVE
# --------------------------------------------------
gmsh.write("five_layers_two_wells.msh")

# Optional visualization
gmsh.fltk.run()

gmsh.finalize()

In [3]:
import os
print(os.getcwd())
print(os.path.exists("five_layers_two_wells.msh"))

C:\Users\dominic.becerra\Documents\OGS\Jupyter notebooks
True


In [1]:
import meshio

mesh = meshio.read("five_layers_two_wells.msh")
meshio.write("mesh.vtu", mesh)

ModuleNotFoundError: No module named 'meshio'